In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

In [2]:
# --- UPDATE THESE PATHS ---
MODEL_PATH      = 'emotion_mobilenetv2_final.keras'
HAAR_CASCADE    = r'haarcascade_frontalface_default.xml'

# To use webcam: set VIDEO_SOURCE = 0
# To use a video file: set VIDEO_SOURCE = r'path\to\your\video.mp4'
VIDEO_SOURCE    = 0

IMG_SIZE        = (96, 96)   # Must match training size
EMOTION_LABELS  = ['Angry', 'Disgusted', 'Fearful', 'Happy', 'Neutral', 'Sad', 'Surprised']

# Confidence threshold — predictions below this show as 'Uncertain'
CONFIDENCE_THRESHOLD = 0.4

In [3]:
print("Loading model...")
emotion_model = load_model(MODEL_PATH)
print("Model loaded successfully.")

face_detector = cv2.CascadeClassifier(HAAR_CASCADE)
if face_detector.empty():
    raise FileNotFoundError(f"Haar cascade not found at: {HAAR_CASCADE}")
print("Face detector loaded.")

Loading model...
Model loaded successfully.
Face detector loaded.


In [4]:
def preprocess_face(face_roi):
    """
    Prepares a detected face ROI for model prediction.
    - Resize to model input size
    - Convert grayscale ROI to 3-channel RGB (model trained on RGB)
    - Normalize to [0, 1]
    - Add batch dimension
    """
    face_resized = cv2.resize(face_roi, IMG_SIZE)

    # If grayscale, convert to 3-channel
    if len(face_resized.shape) == 2:
        face_resized = cv2.cvtColor(face_resized, cv2.COLOR_GRAY2RGB)
    else:
        face_resized = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)

    face_normalized = face_resized.astype('float32') / 255.0
    return np.expand_dims(face_normalized, axis=0)  # Shape: (1, 96, 96, 3)

In [7]:
cap = cv2.VideoCapture(VIDEO_SOURCE)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video source: {VIDEO_SOURCE}")

print("Starting inference. Press Q to quit.")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        print("End of video or camera error.")
        break

    frame = cv2.resize(frame, (1280, 720))
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_detector.detectMultiScale(
        gray_frame,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(30, 30)
    )

    for (x, y, w, h) in faces:
        # Draw bounding box
        cv2.rectangle(frame, (x, y - 50), (x + w, y + h + 10), (0, 255, 0), 2)

        # Extract and preprocess face region
        face_roi = frame[y:y + h, x:x + w]
        processed = preprocess_face(face_roi)

        # Predict
        predictions = emotion_model.predict(processed, verbose=0)
        confidence  = float(np.max(predictions))
        emotion_idx = int(np.argmax(predictions))

        # Apply confidence threshold
        if confidence >= CONFIDENCE_THRESHOLD:
            label = f"{EMOTION_LABELS[emotion_idx]} ({confidence*100:.1f}%)"
            color = (255, 0, 0)
        else:
            label = f"Uncertain ({confidence*100:.1f}%)"
            color = (0, 165, 255)  # Orange for uncertain

        cv2.putText(
            frame, label,
            (x + 5, y - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8, color, 2, cv2.LINE_AA
        )

        # Show confidence bar for all 7 emotions
        bar_x = x + w + 10
        for i, (emotion, prob) in enumerate(zip(EMOTION_LABELS, predictions[0])):
            bar_y = y + i * 25
            bar_len = int(prob * 100)
            bar_color = (0, 255, 0) if i == emotion_idx else (180, 180, 180)
            cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_len, bar_y + 18), bar_color, -1)
            cv2.putText(frame, f"{emotion}: {prob*100:.1f}%",
                        (bar_x + bar_len + 5, bar_y + 14),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

    cv2.imshow('Emotion Detection — MobileNetV2', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Quit by user.")
        break

cap.release()
cv2.destroyAllWindows()
print("Done.")

Starting inference. Press Q to quit.
Quit by user.
Done.
